# Lean DQ Test Run: green_2017 Delta
This notebook runs the DQ wrapper against `bronze.sales.green_2017_test` using `data/green_2017.delta`.

## Objectives
- Load Delta data into a Spark DataFrame with explicit type mapping.
- Execute Great Expectations validation via `run_data_quality`.
- Assert pass/fail contract for a lean regression test.
- Inspect persisted metrics in `logs/dq_metrics.delta`.

## Prerequisites
- Spark and Delta dependencies are available in the active environment.
- The dataset path `data/green_2017.delta` exists relative to the repository root.
- Expectation suite `gx/expectations/bronze.sales.green_2017_test.yml` exists.

## Setup and Initialization
This section configures repository imports for notebook execution and creates or reuses a local Spark session.

The `dq` package cache is cleared so that rerunning cells picks up local source changes without restarting the kernel.

In [1]:
from pathlib import Path
from typing import Any, Callable
import importlib
import sys

from deltalake import DeltaTable
from pyspark.sql import SparkSession

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Reload local dq modules so reruns pick up source edits without a kernel restart.
for name in [mod for mod in list(sys.modules) if mod == "dq" or mod.startswith("dq.")]:
    sys.modules.pop(name)

run_data_quality: Callable[..., dict[str, Any]] = importlib.import_module(
    "dq.dq_runner"
).run_data_quality

spark = SparkSession.getActiveSession() or (
    SparkSession.builder
    .appName("dq-green-2017-test")
    .master("local[*]")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 13:56:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Data Loading and Schema Mapping
Load the Delta table and map each PyArrow column type to a PySpark SQL type.

Unknown Arrow types raise an explicit error to fail fast and avoid silent schema coercion.

In [2]:
import pyarrow as pa
from pyspark.sql.types import (
    BooleanType,
    DataType,
    DateType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

delta_path = repo_root / "data" / "green_2017.delta"
if not delta_path.exists():
    raise FileNotFoundError(f"Missing Delta dataset: {delta_path}")

delta_table = DeltaTable(str(delta_path))
arrow_table = delta_table.to_pyarrow_table()
rows = arrow_table.to_pylist()


def pa_to_spark_type(pa_type: pa.DataType) -> DataType:
    """Convert a PyArrow type to a corresponding PySpark SQL type.

    Args:
        pa_type: Arrow data type from the Delta table schema.

    Returns:
        Matching PySpark SQL DataType.

    Raises:
        ValueError: If the Arrow type is not supported by this mapping.
    """
    if pa.types.is_boolean(pa_type):
        return BooleanType()
    if pa.types.is_integer(pa_type):
        return LongType() if pa_type.bit_width > 32 else IntegerType()
    if pa.types.is_floating(pa_type) or pa.types.is_decimal(pa_type):
        return DoubleType()
    if pa.types.is_timestamp(pa_type):
        return TimestampType()
    if pa.types.is_date(pa_type):
        return DateType()
    if pa.types.is_string(pa_type) or pa.types.is_large_string(pa_type):
        return StringType()
    raise ValueError(f"Unsupported Arrow type for Spark conversion: {pa_type}")

# Build Spark schema from Arrow metadata while keeping fields nullable for ingestion.
spark_schema: StructType = StructType(
    [
        StructField(field.name, pa_to_spark_type(field.type), nullable=True)
        for field in arrow_table.schema
    ]
)

df = spark.createDataFrame(rows, schema=spark_schema)

print("Dataset:", delta_path)
print("Rows loaded:", df.count())

Dataset: /workspaces/great-expectations/data/green_2017.delta


Rows loaded: 1000


## Run Data Quality Validation
Execute the wrapper for `bronze.sales.green_2017_test` and print the returned summary contract.

In [3]:
# Execute DQ validation and capture the normalized summary contract.
result: dict[str, Any] = run_data_quality(
    df=df,
    layer="bronze",
    schema_name="sales",
    table_name="green_2017_test",
    log_metrics=True,
    source_format="delta",
    project_root=repo_root,
)

print(result)

Calculating Metrics: 100%|██████████| 83/83 [00:01<00:00, 46.84it/s]

{'suite_name': 'bronze.sales.green_2017_test', 'run_id': 'b8b86a29-3a21-44ca-b61c-98edc89c85cc', 'success': True, 'evaluated_expectations': 11, 'successful_expectations': 11, 'failed_expectations': 0, 'sampling_strategy': 'statistical', 'original_row_count': 1000, 'sample_row_count': 906, 'confidence': 0.95, 'margin_error': 0.01, 'stratify_by': None, 'metrics_logged': False, 'metrics_rows_written': 0, 'dq_metrics_table': 'analytics.dq_metrics', 'dq_metrics_path': None, 'metrics_sink': 'table', 'metrics_format': 'delta', 'logging_error': 'An error occurred while calling o373.saveAsTable.\n: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: delta. Please find packages at `https://spark.apache.org/third-party-projects.html`.\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:725)\n\tat org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:647)\n\ta

In [4]:
assert result.get("success") is True, f"Validation failed: {result}"
assert result.get("failed_expectations", -1) == 0, (
    f"Expected 0 failed expectations, got {result.get('failed_expectations')}. "
    f"Summary: {result}"
)
print("Test passed: all expectations succeeded.")

Test passed: all expectations succeeded.


## Metrics and Inspection
Read recent records from the metrics Delta table and display them for quick validation diagnostics.

In [ ]:
# Resolve metrics path from notebook or repository-relative locations.
candidate_paths: list[Path] = [
    Path("logs/dq_metrics.delta"),
    Path("../logs/dq_metrics.delta"),
]

metrics_path: Path = next((p for p in candidate_paths if p.exists()), candidate_paths[-1])
metrics_df = DeltaTable(str(metrics_path)).to_pandas().sort_values(
    "run_ts", ascending=False
)

print(f"Metrics Delta path: {metrics_path.resolve()}")
print(f"Total rows: {len(metrics_df)}")
display(metrics_df)

Metrics Delta path: /workspaces/great-expectations/logs/dq_metrics.delta
Total rows: 6


,run_id,run_ts,layer,schema_name,table_name,suite_name,expectation_type,success,unexpected_percent,unexpected_count,element_count,sampling_strategy,original_row_count,sample_row_count,confidence,margin_error,stratify_by,details_json
0,b175b129-d9d4-45c0-9d0d-dbf0fd2fd6dd,2026-03-05 18:03:46.365216+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,NaN,NaN,NaN,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."
1,b175b129-d9d4-45c0-9d0d-dbf0fd2fd6dd,2026-03-05 18:03:46.365216+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,0.0,0.0,10.0,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."
2,b175b129-d9d4-45c0-9d0d-dbf0fd2fd6dd,2026-03-05 18:03:46.365216+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,0.0,0.0,10.0,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."
3,09b713ca-dad2-4798-b515-c05927e55590,2026-03-05 17:31:24.737399+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,NaN,NaN,NaN,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."
4,09b713ca-dad2-4798-b515-c05927e55590,2026-03-05 17:31:24.737399+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,0.0,0.0,10.0,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."
5,09b713ca-dad2-4798-b515-c05927e55590,2026-03-05 17:31:24.737399+00:00,bronze,sales,green_tripdata_2017,bronze.sales.green_tripdata_2017,NaN,True,0.0,0.0,10.0,full,10,10,0.95,0.01,NaN,"{""success"": true, ""expectation_config"": {""type..."


26/03/06 13:56:54 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/03/07 05:32:26 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 251016 ms exceeds timeout 120000 ms
26/03/07 05:32:26 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/07 05:32:36 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointR

## Next Steps
- If validation fails, inspect `result` and the rendered metrics rows for failing expectation details.
- Re-run from Cell 2 when changing local code in `src/dq` to keep imports fresh.
- Promote this notebook pattern for additional bronze test datasets.